# 🎭 LivePortrait — Character Performance Transfer

**Character still + your driving video → the character performs your movements and expressions.**
Runs on the free/Pro Colab T4 — same workflow as your FaceFusion notebook.

## How to use
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. Run cells **1 → 4** top to bottom
3. Cell 2 will ask you to upload your `jobs.zip`

## Job folder format (simpler than FaceFusion — one image + one video per folder)
```
jobs/
  01_terrence_heaven/
      terrence.png        ← character still (any image name: .png .jpg .webp)
      me_performing.mp4   ← your driving video (any video name: .mp4 .mov)
  02_marcus_reply/
      marcus.png
      take3.mp4
  ...
```
Right-click the `jobs` folder → compress to `jobs.zip` → upload in cell 2.
Each folder becomes **one output clip named after the folder** (e.g. `01_terrence_heaven.mp4`).

## Recording your driving video — the key trick
- **Play the ElevenLabs line out loud and mouth along while recording** — the lips will already match
- Phone at eye level, face well lit, whole head in frame
- Front-facing-ish; slightly **exaggerated** expressions transfer better than subtle ones
- Keep body movement minimal — LivePortrait animates face + head, not the body

## Your full free pipeline
1. Generate the line in **ElevenLabs**
2. **Record yourself performing it** (mouth along to the audio)
3. Zip jobs → run this notebook → character performs your movements
4. Drop the clean ElevenLabs audio under the clip in **CapCut**
5. If a mouth is slightly off → run that one clip through your **FaceFusion** notebook with just `lip_syncer`

> **Test first with one job** — Terrence's *"Heaven, where you at?"* (4 words). You'll know in ~10 minutes
> whether the quality works for you before committing to all 14 shots.


In [ ]:
#@title 1️⃣ Setup — install LivePortrait + download models (~4–6 min, run once per session)
import subprocess, os

# Make sure we're on a GPU runtime
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU found. Runtime → Change runtime type → T4 GPU, then re-run this cell.")
print("GPU:", gpu.stdout.strip())

%cd /content
if not os.path.exists("/content/LivePortrait"):
    !git clone -q https://github.com/KwaiVGI/LivePortrait.git
%cd /content/LivePortrait

# Install requirements, but keep Colab's own torch build (faster, avoids CUDA mismatches)
!grep -ivE "^(torch|torchvision|torchaudio)" requirements.txt > reqs_colab.txt
!pip install -q -r reqs_colab.txt
!pip install -q "huggingface_hub[cli]"

# Download the pretrained weights (~1.5 GB)
!huggingface-cli download KwaiVGI/LivePortrait --local-dir pretrained_weights --exclude "*.git*" "README.md" "docs"

print("\n✅ Setup complete — go to cell 2")


In [ ]:
#@title 2️⃣ Upload your jobs.zip
import os, shutil, zipfile
from google.colab import files

JOBS_DIR = "/content/jobs"
if os.path.exists(JOBS_DIR):
    shutil.rmtree(JOBS_DIR)
os.makedirs(JOBS_DIR)

print("Pick your jobs.zip …")
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall(JOBS_DIR)
os.remove(zip_name)

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp"}
VIDEO_EXT = {".mp4", ".mov", ".m4v", ".webm", ".avi"}

def find_jobs(root):
    jobs = []
    for dirpath, dirnames, filenames in os.walk(root):
        if "__MACOSX" in dirpath:
            continue
        filenames = [f for f in filenames if not f.startswith((".", "._"))]
        imgs = [f for f in filenames if os.path.splitext(f)[1].lower() in IMAGE_EXT]
        vids = [f for f in filenames if os.path.splitext(f)[1].lower() in VIDEO_EXT]
        if imgs and vids:
            jobs.append({
                "name": os.path.basename(dirpath),
                "image": os.path.join(dirpath, sorted(imgs)[0]),
                "video": os.path.join(dirpath, sorted(vids)[0]),
            })
    return sorted(jobs, key=lambda j: j["name"])

jobs = find_jobs(JOBS_DIR)
if not jobs:
    raise RuntimeError(
        "No job folders found. Each job folder needs one image (character still) "
        "and one video (your driving performance). Re-zip and upload again."
    )

print(f"\nFound {len(jobs)} job(s):")
for j in jobs:
    print(f"  • {j['name']}:  {os.path.basename(j['image'])}  +  {os.path.basename(j['video'])}")
print("\n✅ Looks good? Go to cell 3")


In [ ]:
#@title 3️⃣ Animate every job (~1–3 min per job on a T4)
CROP_DRIVING_VIDEO = True     #@param {type:"boolean"}
EXPRESSION_STRENGTH = 1.0     #@param {type:"slider", min:0.5, max:1.5, step:0.05}
FORCE_RERUN = False           #@param {type:"boolean"}

# CROP_DRIVING_VIDEO: leave ON for normal phone recordings (auto-crops to your face).
#   Only turn OFF if your driving video is already a tight face crop.
# EXPRESSION_STRENGTH: 1.0 = copy you exactly; >1.0 exaggerates, <1.0 tones it down.
# FORCE_RERUN: re-animate jobs that already have a result (e.g. after changing settings).

import os, glob, shutil, subprocess, time

RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

done, failed = [], []
for i, job in enumerate(jobs, 1):
    out_path = os.path.join(RESULTS_DIR, f"{job['name']}.mp4")
    if os.path.exists(out_path) and not FORCE_RERUN:
        print(f"[{i}/{len(jobs)}] {job['name']} — already done, skipping")
        done.append(job["name"])
        continue

    print(f"[{i}/{len(jobs)}] {job['name']} — animating…")
    work_dir = f"/content/work/{job['name']}"
    shutil.rmtree(work_dir, ignore_errors=True)

    cmd = ["python", "inference.py",
           "-s", job["image"],
           "-d", job["video"],
           "--output_dir", work_dir,
           "--driving_multiplier", str(EXPRESSION_STRENGTH)]
    if CROP_DRIVING_VIDEO:
        cmd.append("--flag_crop_driving_video")

    t0 = time.time()
    r = subprocess.run(cmd, cwd="/content/LivePortrait", capture_output=True, text=True)
    outputs = [p for p in glob.glob(os.path.join(work_dir, "*.mp4"))
               if "concat" not in os.path.basename(p)]

    if r.returncode == 0 and outputs:
        shutil.copy(outputs[0], out_path)
        print(f"    ✅ done in {time.time() - t0:.0f}s → results/{job['name']}.mp4")
        done.append(job["name"])
    else:
        print("    ❌ failed — last lines of the log:")
        log = (r.stderr or r.stdout or "").strip().splitlines()[-8:]
        for line in log:
            print("      " + line)
        failed.append(job["name"])

print(f"\nFinished: {len(done)} ok, {len(failed)} failed")
if failed:
    print("Failed:", ", ".join(failed), "— see logs above, or check Troubleshooting at the bottom")
print("✅ Go to cell 4 to download")


In [ ]:
#@title 4️⃣ Zip + download all results
import os, shutil
from google.colab import files

RESULTS_DIR = "/content/results"
clips = sorted(f for f in os.listdir(RESULTS_DIR) if f.endswith(".mp4"))
if not clips:
    raise RuntimeError("No results yet — run cell 3 first.")

print(f"Packaging {len(clips)} clip(s):")
for c in clips:
    print("  •", c)

shutil.make_archive("/content/liveportrait_results", "zip", RESULTS_DIR)
files.download("/content/liveportrait_results.zip")
print("\n✅ Done — drop the clips into CapCut and lay the clean ElevenLabs audio under each one")


## 🔧 Troubleshooting

| Problem | Fix |
|---|---|
| *"No GPU found"* in cell 1 | Runtime → Change runtime type → **T4 GPU**, then re-run cell 1 |
| Face not detected in the still | Use a clearer, front-facing character still — face should be reasonably large in frame |
| Face not detected in your video | Better lighting, face the camera, keep your whole head in frame; keep `CROP_DRIVING_VIDEO` on |
| Out of memory / crash | Use shorter driving clips (one line per job, not a whole monologue) |
| Mouth slightly off | Expected sometimes — run that one clip through your **FaceFusion notebook with just `lip_syncer`** |
| Movement too subtle / too wild | Re-run with `EXPRESSION_STRENGTH` at 1.15 (stronger) or 0.85 (calmer) + `FORCE_RERUN` on |
| Body doesn't move | That's the tool's limit — LivePortrait animates **face + head only**. For real body movement (the 5B throw, walking shots), use Flow/Runway generation |

**Colab disconnected?** Re-run cell 1 (setup) and cell 2 (upload), then cell 3 — already-finished jobs are skipped automatically, so nothing is lost except unfinished ones.
